# 18 - Where does federation help? Low-data curve and a new hospital

Pre-registered in `docs/2026-10-28-low-data-and-new-site-preregistration.md` (read it first). Both parts use
**10 hospitals from `dirichlet_ward_mixture(alpha=0.5)`**: every hospital sees the same organisms, hospitals differ
in ward mix, and the partition changes with the seed (five distinct partitions, seeds 42-46). Rounds = 8, local epochs = 4,
local-only epochs = 32, hidden = 128. Same validation-only protocol as notebooks 16/17: every hospital picks its own
cutoff on its own validation rows, test is read once.

**Part A - low-data curve.** Every hospital keeps a fraction f = 1.0 / 0.25 / 0.10 / 0.03 of its train **and validation**
patients; its test patients never change. Arms per level: local-only, FedAvg, FedAvg + fine-tune (same seed, one run).
Primary metric: the **unweighted mean over hospitals** of test macro-F1.

**Part B - new hospital.** Hold one hospital out (the 5th largest = `median`, and the `smallest`), train FedAvg on the other nine,
then give the held-out hospital k = 25 / 100 / 400 / all labelled training patients and compare: the federated model as is
(**zero-shot**), the federated model **fine-tuned** on those patients, and a model **trained from scratch** on the same patients.
Test = the hospital's full test set.

Run Part A and Part B in separate cells (cell 4 and cell 5) - on two accounts if you like: set `TAG` and merge with cell 9.
Results: `results/amr_regimes_<TAG>.json` (checkpointed after every seed, resumed if present, mirrored to Drive on Colab).
This is a test of the hypothesis that federation helps most when local data are scarce; a tie or a loss at any level is reported as such.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this is the final-protocol branch (modules below must exist)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import final_protocol as P, analysis as A, selection, regimes, regime_runs as RR, new_site
from amr_fed.topology import personalized_weights, train_frame
print('imports OK | v3 temperatures:', P.T_GRID, '| fine-tune epochs:', P.FT_EPOCHS)

In [ ]:
# 3) Load cohort once + smoke: 10 hospitals per seed, a different partition per seed
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import dirichlet_ward_mixture
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS:
    a = dirichlet_ward_mixture(df, n_clients=10, alpha=0.5, seed=s)
    print(f'seed {s}: {a.nunique()} hospitals | patients (largest first): {a.value_counts().sort_values(ascending=False).tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'
print('SMOKE PASS')

In [ ]:
# 4) PART A - low-data curve (resume-safe; 4 levels x 5 seeds = 20 FedAvg runs, each with local-only + fine-tune)
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed') and not m.startswith('amr_fed.federated.final') and not m.startswith('amr_fed.federated.regime')]:
    del _s.modules[_m]
from amr_fed.federated import final_protocol as P, analysis as A, regime_runs as RR
from amr_fed.partition import dirichlet_ward_mixture
TAG = 'all'
KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: dirichlet_ward_mixture(d, n_clients=10, alpha=0.5, seed=s),
          n_clients=10, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RESULTS = SRC.parent / 'results' / f'amr_regimes_{TAG}.json'
RESULTS.parent.mkdir(exist_ok=True)
MIRROR = Path(f'/content/drive/MyDrive/amr_regimes_{TAG}.json') if ON_COLAB else None
src = RESULTS if RESULTS.exists() else (MIRROR if MIRROR and MIRROR.exists() else None)
res = json.load(open(src)) if src else {}
ck = lambda r: P.save_json(r, RESULTS, MIRROR)
res = RR.run_scarcity(res, kw=KW, checkpoint=ck, ray_shutdown=ray.shutdown)
P.save_json(res, RESULTS, MIRROR)
print('part A done:', [k for k in res if k.startswith('scarcity')])

In [ ]:
# 5) PART B - new hospital (resume-safe; 2 hold-out rules x 5 seeds = 10 federated runs on 9 hospitals + cheap adaptation runs)
res = RR.run_newsite(res, kw=KW, checkpoint=ck, ray_shutdown=ray.shutdown)
P.save_json(res, RESULTS, MIRROR)
print('part B done:', [k for k in res if k.startswith('newsite')])

In [ ]:
# 6) PART A results: local-only vs FedAvg vs FedAvg + fine-tune as data shrink (primary = unweighted mean over hospitals)
print(A.scarcity_report(res))

In [ ]:
# 7) PART B results: zero-shot vs fine-tuned vs from-scratch at a new hospital
print(A.newsite_report(res))

In [ ]:
# 8) Compute / provenance: wall-clock per part and the environment
import numpy as np
for k, v in res.items():
    if k.startswith('scarcity'):
        print(k, 'min per seed (mean):', round(np.mean([r['wall_seconds'] for r in v['runs']]) / 60, 1), '| gpu:', v['runs'][0]['provenance'].get('gpu'), '| git:', v['runs'][0]['provenance'].get('git'))
    if k.startswith('newsite'):
        print(k, 'min per seed (federated part, mean):', round(np.mean([r['fed_wall_seconds'] for r in v['runs']]) / 60, 1))

In [ ]:
# 9) (Optional) merge files from two accounts, then re-run cells 6-8 with res = merged
import glob
parts = sorted(glob.glob(str(SRC.parent / 'results' / 'amr_regimes_[!m]*.json')))
if ON_COLAB: parts += sorted(glob.glob('/content/drive/MyDrive/amr_regimes_[!m]*.json'))
print('files:', parts)
merged = P.merge_results(parts)
P.save_json(merged, SRC.parent / 'results' / 'amr_regimes_merged.json')
print('merged keys:', [k for k in merged if not k.startswith('_')])
# to analyse the merged set:  res = merged